# Providers 02 - Bedrock Runtime

Objetivo: comprobar `bedrock-runtime` mediante la misma fachada publica usada
por otros providers y distinguir sus dos rutas de autenticacion.

**Lugar en el modelo:** Bedrock es el Provider (donde corre la inferencia);
credenciales, region y modelo son configuracion, no logica del Agent.

**Evidencia exigida:** con configuracion valida, la ejecucion debe producir un
`RunResult` exitoso con `engine="bedrock-runtime"`; sin ella debe reportar
`not-run` con motivo.

**Límite de la evidencia:** el snapshot prueba deteccion local, no permisos ni
inferencia. Solo una fila `passed` contiene evidencia live.


## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_BEDROCK_LIVE | 1 | Usa 0 para desactivar la llamada real. |
| BEDROCK_MODEL_ID | provider default | Modelo habilitado en la cuenta y region. |
| AWS_REGION o AWS_DEFAULT_REGION | us-east-1 | Region de Bedrock. |
| cadena AWS estandar | auto | ADA, perfil, rol/ARN, web identity o credenciales temporales SigV4. |
| AWS_BEARER_TOKEN_BEDROCK | sin valor | API key temporal nativa de Bedrock fuera o dentro de AWS. |

### Dos rutas, un solo Provider

- **ADA/IAM/ARN:** boto3 resuelve la cadena AWS normal, firma con SigV4 y puede
  consultar identidad STS cuando la politica lo permite.
- **API key temporal:** boto3 lee `AWS_BEARER_TOKEN_BEDROCK` y autentica la
  llamada Bedrock como Bearer. Esa credencial es regional, dura como maximo la
  sesion temporal (hasta 12 horas) y no autentica STS.
- Si ambas señales estan presentes, Agentic Systems reporta
  `authentication_mode="bedrock-api-key"` y no intenta `GetCallerIdentity`.
- Las dos rutas convergen en `bedrock-runtime` y deben producir el mismo
  contrato `RunResult`.


## Contrato de la demostracion

El notebook prueba el provider Unicamente a traves de la fachada publica:

```text
toolkit.runtime - toolkit.system - system.agent - RunResult
```

La celda live se habilita con una variable explicita. Sin credenciales o endpoint, el notebook permanece ejecutable y muestra un estado not-run estructurado. Cuando la variable esta activa, cualquier error real del provider debe ser visible.

In [ ]:
import os

import agentic_systems as toolkit

aws_environment = toolkit.aws_environment_snapshot()

RUN_BEDROCK_LIVE = os.getenv("RUN_BEDROCK_LIVE", "1").strip().lower() in {"1", "true", "yes"}
REGION = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
MODEL = os.getenv("BEDROCK_MODEL_ID")
AGENT_NAME = "bedrock_public_api_probe"

aws_session = toolkit.boto3_session_snapshot(region_name=REGION)
HAS_BEARER = bool(os.getenv("AWS_BEARER_TOKEN_BEDROCK"))
AUTH_MODE = aws_session.get("authentication_mode")
assert AUTH_MODE in {None, "aws-credential-chain", "bedrock-api-key"}

toolkit.show_json({
    "package": toolkit.__name__,
    "version": toolkit.__version__,
    "run_live": RUN_BEDROCK_LIVE,
    "environment": aws_environment,
    "session": aws_session,
    "bearer_token_configured": HAS_BEARER,
}, title="Preflight Bedrock")

## 1) Declarar runtime y limites

Modelo y region proceden del ambiente o de los defaults publicos. El notebook no fija un modelo o region como verdad universal.

In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=90,
    max_retries=1,
    max_tool_calls=1,
    max_turns=3,
    max_concurrency=1,
)

runtime = toolkit.runtime(
    provider="bedrock-runtime",
    model=MODEL,
    region=REGION,
    scheduler=scheduler,
    metadata={"tutorial": "providers/bedrock"},
)

toolkit.show_json(runtime.describe(), title="Bedrock RuntimeConfig")

## 2) Crear system, tool y agent

Ni el agente ni la tool conocen boto3. Cambiar de provider no cambia su contrato.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    """Verifica un simbolo contra la superficie publica instalada."""
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name=AGENT_NAME,
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Responde con el nombre, si es publico y la version observada."
    ),
    tools=[inspect_public_api],
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        trace="compact",
        strict=True,
    ),
)

toolkit.show_json(agent.info(), title="Agente declarado")

## 3) Ejecutar o reportar not-run

La llamada live esta habilitada por defecto cuando existe region y una de las
dos rutas de autenticacion. En ADA se espera la cadena AWS/rol; fuera de AWS se
puede usar la API key temporal. Usa `RUN_BEDROCK_LIVE=0` para forzar
`not-run`. Errores de permisos, region, expiracion o modelo permanecen
visibles.


In [ ]:
can_run = RUN_BEDROCK_LIVE and (
    bool(aws_session.get("has_credentials")) or HAS_BEARER
)

if can_run:
    result = agent.run(
        "Verifica si system pertenece a la API publica instalada.",
        mode="eval",
    )
    assert isinstance(result, toolkit.RunResult)
    assert result.ok, result.errors
    assert result.engine == "bedrock-runtime"
    toolkit.human_result(result, title="Bedrock RunResult", show_lineage=True)
    toolkit.show_json(toolkit.run_result_output(result), title="Contrato normalizado")
else:
    result = None
    toolkit.show_json({
        "status": "not-run",
        "provider": "bedrock-runtime",
        "reason": "Configura credenciales AWS SigV4 o AWS_BEARER_TOKEN_BEDROCK, o usa RUN_BEDROCK_LIVE=0.",
    }, title="Bedrock live gate")

## 4) API realmente ejercitada

Diagnostico AWS y ejecucion usan exclusivamente funciones publicas de `toolkit`.

In [ ]:
api_coverage = [
    "toolkit.aws_environment_snapshot",
    "toolkit.boto3_session_snapshot",
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.RunResult",
    "toolkit.show_json",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
]

toolkit.show_json(api_coverage, title="Bedrock API coverage")

## Resultado e interpretacion

Con live desactivado: snapshot seguro, modo de autenticacion y configuracion
observable. Con live activado: un `RunResult` real cuyo runtime reporta
`bedrock-runtime`. El modo API key no promete identidad STS; el modo
IAM/ARN si puede reportarla, pero ambos conservan el mismo contrato de
ejecucion y nunca exponen secretos.
